In [ ]:
import random 
import os

In [ ]:
def read_fasta(file_path, max_seqs=1000000):
    sequences = {}
    current_header = None
    current_sequence = []

    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if current_header is not None:
                    sequences[current_header] = "".join(current_sequence)
                    current_sequence = []
                current_header = line[1:] 
            else:
                current_sequence.append(line)
            if len(sequences) > max_seqs:
                break
        if current_header is not None:
            sequences[current_header] = "".join(current_sequence)

    return sequences

all_seqs = read_fasta("../../outputs/petase/sequence.fasta")
len(all_seqs)

In [ ]:
random.seed(0)
all_seqs_name = list(all_seqs.keys())
random.shuffle(all_seqs_name)
seq_pool_name = all_seqs_name[:1000]

In [ ]:
os.makedirs("../../outputs/petase/sample1000_search", exist_ok=True)
with open("../../outputs/petase/sample1000_search/seq_pool.fasta", 'w') as f:
    for name in seq_pool_name:
        f.write(f">{name}\n{all_seqs[name]}\n")

## 1.  Self-search within the 1,000-sequence pool

In [ ]:
%%bash

input_path="../../outputs/petase/sample1000_search/seq_pool.fasta"

target_path="../../outputs/petase/sample1000_search/seq_pool.fasta"

output_path="${input_path%.fasta}_mmseqs_hit.tsv"

echo "output to $output_path"

mmseqs easy-search "$input_path" "$target_path" "$output_path" /tmp \
      --min-seq-id 0.5 \
      -c 0.8 \
      --cov-mode 0 \
      --format-output query,target,pident,fident,nident,alnlen,qcov,tcov \
      --threads 96

In [ ]:
## see compare results
from collections import defaultdict
res_file = "../../outputs/petase/sample1000_search/seq_pool_mmseqs_hit.tsv"
name2sim = defaultdict(list)
with open(res_file, 'r') as f:
    for line in f:
        name, sim_name = line.strip().split("\t")[:2]
        name2sim[name].append(sim_name)
len(name2sim)

In [ ]:
## single found count (only itself)
sum([1 if len(v)==1 else 0 for v in name2sim.values()])

## 2. Cluster the sequence pool and count clusters

In [ ]:
%%bash

input_path="../../outputs/petase/sample1000_search/seq_pool.fasta"

cluster_prefix="${input_path%.fasta}_mmseqs_cluster"

echo "cluster prefix: $cluster_prefix"

mmseqs easy-cluster "$input_path" "$cluster_prefix" /tmp \
      --min-seq-id 0.5 \
      -c 0.8 \
      --cov-mode 0 \
      --threads 96

In [ ]:
## read res 
rep_set = set()
with open("../../outputs/petase/sample1000_search/seq_pool_mmseqs_cluster_cluster.tsv", 'r') as f:
    for line in f:
        rep, entry = line.strip().split("\t")
        rep_set.add(rep)

f"cluster number: {len(rep_set)}"